<a href="https://colab.research.google.com/github/sreechaturya-github/AI_litter_detection/blob/main/nb/Gemma4_(E4B)-Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a Google Colab L4 instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E4B-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

We now add LoRA adapters for parameter efficient fine-tuning, allowing us to train only 1% of all model parameters efficiently.

**[NEW]** We also support fine-tuning only the vision component, only the language component, or both. Additionally, you can choose to fine-tune the attention modules, the MLP layers, or both!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 32,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 32,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We'll use a sampled dataset of handwritten math formulas. The objective is to convert these images into a computer-readable format—specifically LaTeX—so they can be rendered. This is particularly useful for complex expressions.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [6]:
import os
os.makedirs("/content/litter_text_dataset/images", exist_ok=True)


In [7]:
from google.colab import files
uploaded = files.upload()


Saving unsloth_vision_chat_colab.jsonl to unsloth_vision_chat_colab.jsonl


In [8]:
import shutil
shutil.move(
    "/content/unsloth_vision_chat_colab.jsonl",
    "/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl",
)


'/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl'

In [9]:
from google.colab import files
uploaded = files.upload()


Saving images 2.zip to images 2.zip


In [15]:
import zipfile

with zipfile.ZipFile("/content/images 2.zip", "r") as zip_ref:
    zip_ref.extractall("/content/litter_text_dataset/images")



In [16]:
import os

print("JSONL exists:", os.path.exists("/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl"))
print("Images folder exists:", os.path.exists("/content/litter_text_dataset/images"))
print("Some images:", os.listdir("/content/litter_text_dataset/images")[:5])



JSONL exists: True
Images folder exists: True
Some images: ['images.zip', '__MACOSX']


In [17]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl",
    split="train",
)

train_dataset = dataset.filter(lambda x: x["split"] == "train")
val_dataset = dataset.filter(lambda x: x["split"] == "val")

print(train_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/18 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18 [00:00<?, ? examples/s]

{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/litter_text_dataset/images/v1_0p0s.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Classify this CCTV frame as litter, no_litter, or cleaning, and describe what is visible.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Label: no_litter. Description: A rainy street scene with pedestrians and a large truck. The ground looks wet, but there is no clear unmanaged litter or cleanup activity visible.'}]}], 'label': 'no_litter', 'split': 'train', 'source_video': 'v1.mp4', 'timestamp_seconds': 0.0, 'id': 'v1_0p0s'}


Let's take an overview of the dataset. We'll examine the second image and its corresponding caption.

In [19]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl",
    split="train",
)

train_dataset = dataset.filter(lambda x: x["split"] == "train")
val_dataset = dataset.filter(lambda x: x["split"] == "val")

print(train_dataset)
print(train_dataset[0])


Dataset({
    features: ['messages', 'label', 'split', 'source_video', 'timestamp_seconds', 'id'],
    num_rows: 14
})
{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/litter_text_dataset/images/v1_0p0s.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Classify this CCTV frame as litter, no_litter, or cleaning, and describe what is visible.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Label: no_litter. Description: A rainy street scene with pedestrians and a large truck. The ground looks wet, but there is no clear unmanaged litter or cleanup activity visible.'}]}], 'label': 'no_litter', 'split': 'train', 'source_video': 'v1.mp4', 'timestamp_seconds': 0.0, 'id': 'v1_0p0s'}


In [20]:
converted_dataset = train_dataset
eval_dataset = val_dataset


In [21]:
print(converted_dataset[0]["messages"])


[{'role': 'user', 'content': [{'type': 'image', 'image': '/content/litter_text_dataset/images/v1_0p0s.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Classify this CCTV frame as litter, no_litter, or cleaning, and describe what is visible.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Label: no_litter. Description: A rainy street scene with pedestrians and a large truck. The ground looks wet, but there is no clear unmanaged litter or cleanup activity visible.'}]}]


In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-4"
)

Before fine-tuning, let us evaluate the base model's performance. We do not expect strong results, as it has not encountered this chat template before.

In [26]:
import os
import shutil

src_dir = "/content/litter_text_dataset/images/images.zip"
dst_dir = "/content/litter_text_dataset/images"

for name in os.listdir(src_dir):
    src = os.path.join(src_dir, name)
    dst = os.path.join(dst_dir, name)
    if os.path.isfile(src) and name.lower().endswith(".jpg"):
        shutil.move(src, dst)

print("Moved jpg files to:", dst_dir)


Moved jpg files to: /content/litter_text_dataset/images


In [27]:
import os

print(os.path.exists("/content/litter_text_dataset/images/v1_0p0s.jpg"))
print(os.listdir("/content/litter_text_dataset/images")[:10])


True
['v9_5p0s.jpg', 'v1_4p0s.jpg', 'v5_0p0s.jpg', 'images.zip', 'v2_0p0s.jpg', '__MACOSX', 'v8_0p0s.jpg', 'v7_0p0s.jpg', 'v3_0p0s.jpg', 'v3_5p0s.jpg']


In [28]:
from PIL import Image

sample = converted_dataset[0]
image_path = sample["messages"][0]["content"][0]["image"]
instruction = sample["messages"][0]["content"][1]["text"]

image = Image.open(image_path).convert("RGB")
print(image_path)


/content/litter_text_dataset/images/v1_0p0s.jpg


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [31]:
import sys
print(sys.executable)


/usr/bin/python3


In [33]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-p76hhtj3/unsloth_5261a883cf3a4450bf62f4fbb95bd1cb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-p76hhtj3/unsloth_5261a883cf3a4450bf62f4fbb95bd1cb
  Resolved https://github.com/unslothai/unsloth.git to commit 13928b5f0ed215667f8c54abe1b975bbd6ab5ce5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 69.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 74.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [34]:
import unsloth
from unsloth import FastVisionModel
print("Unsloth is ready")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth is ready


In [35]:
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print(train_dataset[0]["messages"])


Train: 14
Val: 4
[{'role': 'user', 'content': [{'type': 'image', 'image': '/content/litter_text_dataset/images/v1_0p0s.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Classify this CCTV frame as litter, no_litter, or cleaning, and describe what is visible.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Label: no_litter. Description: A rainy street scene with pedestrians and a large truck. The ground looks wet, but there is no clear unmanaged litter or cleanup activity visible.'}]}]


In [ ]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
from unsloth import FastVisionModel
import torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
model, processor = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [5]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = True,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)


In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/litter_text_dataset/unsloth_vision_chat_colab.jsonl",
    split="train",
)

train_dataset = dataset.filter(lambda x: x["split"] == "train")
val_dataset = dataset.filter(lambda x: x["split"] == "val")

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print(train_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/18 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18 [00:00<?, ? examples/s]

Train: 14
Val: 4
{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/litter_text_dataset/images/v1_0p0s.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Classify this CCTV frame as litter, no_litter, or cleaning, and describe what is visible.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Label: no_litter. Description: A rainy street scene with pedestrians and a large truck. The ground looks wet, but there is no clear unmanaged litter or cleanup activity visible.'}]}], 'label': 'no_litter', 'split': 'train', 'source_video': 'v1.mp4', 'timestamp_seconds': 0.0, 'id': 'v1_0p0s'}


In [9]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

converted_dataset = train_dataset
eval_dataset = val_dataset

trainer = SFTTrainer(
    model = model,
    train_dataset = converted_dataset,
    eval_dataset = eval_dataset,
    processing_class = processor,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [10]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,3.471920
2,3.600998
3,3.636336
4,7.257757
5,3.608925
6,3.583045
7,3.121605
8,4.221947
9,2.224359
10,1.810005


In [11]:
from PIL import Image
from transformers import TextStreamer

sample = val_dataset[0]
image_path = sample["messages"][0]["content"][0]["image"]
instruction = sample["messages"][0]["content"][1]["text"]

image = Image.open(image_path).convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": instruction},
        ],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(processor.tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)


Classification: **cleaning**

Description: The image shows a person wearing an orange jumpsuit standing on a gray floor. They are holding two large, blue plastic bags, suggesting they are engaged in a cleaning or cleanup activity. There is no visible litter in the scene.<turn|>


In [12]:
model.save_pretrained("gemma_4_litter_lora")
processor.save_pretrained("gemma_4_litter_lora")


['gemma_4_litter_lora/processor_config.json']

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
10.182 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,686 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 82,444,288 of 8,078,600,736 (1.02% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,14.862482
2,15.983265
3,15.897610
4,16.153982
5,16.364323
6,12.335323
7,6.774734
8,5.239524
9,4.966103
10,3.271874


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

353.8392 seconds used for training.
5.9 minutes used for training.
Peak reserved memory = 10.777 GB.
Peak reserved memory for training = 0.595 GB.
Peak reserved memory % of max memory = 74.003 %.
Peak reserved memory for training % of max memory = 4.086 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can modify the instruction and input—just leave the output blank.

We'll use the best hyperparameters for inference on Gemma: `top_p=0.95`, `top_k=64`, and `temperature=1.0`.

In [ ]:
image = dataset[10]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt = True)

inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

\[\left[[B_n^{\pm}, b_2^{\mp}], b_2^{\pm}\right] = nB_n^{\pm}, \quad \left[[B_n^{\mp}, b_2^{\pm}], b_2^{\mp}\right] = nB_n^{\mp}.\]<turn|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma_4_lora")  # Local saving
processor.save_pretrained("gemma_4_lora")
# model.push_to_hub("your_name/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# processor.push_to_hub("your_name/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

['gemma_4_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastVisionModel

    model, processor = FastVisionModel.from_pretrained(
        model_name = "gemma_4_lora",  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True,  # Set to False for 16bit LoRA
    )

sample = dataset[1]
image = sample["image"].convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": sample["text"],
            },
            {
                "type": "image",
            },
        ],
    },
]
input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor.tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

This equation appears to be from a field of physics, likely **theoretical physics**, specifically **electrodynamics**, **gauge theories**, or **relativistic field theory**, given the use of Greek letters ($\mu, \alpha, \beta$), tensor notation ($D^\alpha_\mu \tilde{A}^\alpha_\mu$), and the structure of the equation.

Here is a breakdown of the components and the likely physical meaning:

### Breakdown of the Equation

$$D^\alpha_\mu \tilde{A}^\alpha_\mu = 0$$

1.  **$D^\alpha_\mu$ (Covariant Derivative or Differential


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", processor,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", processor, token = "YOUR_HF_TOKEN")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).